
# NBA Shot Quality — End-to-End Classification + Expected Points (Fixed One‑Hot Strategy)

This notebook:
1. Loads and merges your 2015–2016 shot-by-shot CSVs.
2. Keeps a copy of `player_name` **before** encoding.
3. One‑hot encodes categorical features **excluding** `name`.
4. Trains a baseline classifier (Gradient Boosting) to predict `shot_made_flag`.
5. Derives `shot_points` **from the one‑hot** `shot_type_3PT Field Goal` dummy (original strategy).
6. Computes per‑shot expected points, aggregates to player‑level **luck** (actual − expected).
7. Saves outputs (luck table CSV) for download.


In [5]:

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# File paths (already uploaded to /mnt/data)
files = [
    "nba_savant 2015-2016 a-h.csv",
    "nba_savant 2015-2016 i-m.csv",
    "nba_savant 2015-2016 n-z.csv",
]

files


['nba_savant 2015-2016 a-h.csv',
 'nba_savant 2015-2016 i-m.csv',
 'nba_savant 2015-2016 n-z.csv']

In [6]:

dfs = []
for f in files:
    assert os.path.exists(f), f"Missing file: {f}"
    dfs.append(pd.read_csv(f))

df = pd.concat(dfs, ignore_index=True)
print("Merged shape:", df.shape)
df.head()


Merged shape: (145986, 22)


,name,team_name,game_date,season,espn_player_id,team_id,espn_game_id,period,minutes_remaining,seconds_remaining,...,shot_type,shot_distance,opponent,x,y,dribbles,touch_time,defender_name,defender_distance,shot_clock
0,Clint Capela,Houston Rockets,2016-01-24,2015,3102529.0,1610612745,400828552,4,1,26,...,2PT Field Goal,0,Dallas Mavericks,0,1,0,0.0,NaN,0.0,0.0
1,Clint Capela,Houston Rockets,2015-10-28,2015,3102529.0,1610612745,400827897,2,4,50,...,2PT Field Goal,3,Denver Nuggets,-25,31,0,0.0,"Mudiay, Emmanuel",2.7,12.9
2,Tim Hardaway Jr,Atlanta Hawks,2016-01-09,2015,2528210.0,1610612737,400828438,1,2,0,...,2PT Field Goal,0,Chicago Bulls,0,1,0,0.0,"Butler, Jimmy",4.9,20.4
3,Andre Drummond,Detroit Pistons,2015-10-28,2015,6585.0,1610612765,400827894,3,2,28,...,2PT Field Goal,3,Utah Jazz,18,26,0,0.0,"Favors, Derrick",3.3,0.0
4,Kenneth Faried,Denver Nuggets,2016-02-08,2015,6433.0,1610612743,400828665,2,4,1,...,2PT Field Goal,0,Brooklyn Nets,0,1,0,0.0,NaN,0.0,0.0


In [7]:

# Keep copy for grouping later (do not encode this column)
df["player_name"] = df["name"]

# Target
target_col = "shot_made_flag"
assert target_col in df.columns, f"Missing target column: {target_col}"

# Optional: drop clear identifiers not used as features
drop_cols_if_present = ["game_date", "season", "espn_player_id", "team_id", "espn_game_id"]
for c in drop_cols_if_present:
    if c in df.columns:
        pass  # keep them around for now; safe to drop later if desired

# Show columns
df.columns.tolist()


['name',
 'team_name',
 'game_date',
 'season',
 'espn_player_id',
 'team_id',
 'espn_game_id',
 'period',
 'minutes_remaining',
 'seconds_remaining',
 'shot_made_flag',
 'action_type',
 'shot_type',
 'shot_distance',
 'opponent',
 'x',
 'y',
 'dribbles',
 'touch_time',
 'defender_name',
 'defender_distance',
 'shot_clock',
 'player_name']

In [8]:

# Define columns based on your sample schema
categorical_cols = ["team_name", "opponent", "action_type", "shot_type", "defender_name"]
numeric_cols = [
    "period", "minutes_remaining", "seconds_remaining",
    "shot_distance", "x", "y", "dribbles", "touch_time",
    "defender_distance", "shot_clock"
]

# Safety: ensure these columns exist
missing_cats = [c for c in categorical_cols if c not in df.columns]
missing_nums = [c for c in numeric_cols if c not in df.columns]
if missing_cats or missing_nums:
    print("Note: Some expected columns are missing. "
          f"Missing categoricals: {missing_cats}, missing numerics: {missing_nums}")

# Fill categorical NAs to a string
for c in categorical_cols:
    if c in df.columns:
        df[c] = df[c].fillna("Unknown")

# Coerce numerics (strings like '00' → 0; NAs → 0)
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

# Confirm dtypes
df[numeric_cols].dtypes


period                 int64
minutes_remaining      int64
seconds_remaining      int64
shot_distance          int64
x                      int64
y                      int64
dribbles               int64
touch_time           float64
defender_distance    float64
shot_clock           float64
dtype: object

In [9]:

df_enc = pd.get_dummies(df, columns=[c for c in categorical_cols if c in df.columns], drop_first=True)

# Build X (numeric + encoded categorical) and y
encoded_cols = [c for c in df_enc.columns if any((f"{base}_" in c) for base in categorical_cols)]
X_cols = [c for c in numeric_cols if c in df_enc.columns] + encoded_cols

# Ensure only numeric in X
X = df_enc[X_cols].apply(pd.to_numeric, errors="raise")
y = df_enc[target_col].astype(int)

X.shape, y.shape


((145986, 563), (145986,))

In [10]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = GradientBoostingClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"GradientBoost: acc={accuracy_score(y_test, y_pred):.3f}, auc={roc_auc_score(y_test, y_proba):.3f}")


In [ ]:

# Use one‑hot 3PT dummy to derive shot_points (original strategy)
three_dummy_candidates = [c for c in df_enc.columns if c.startswith("shot_type_") and "3PT" in c]

if len(three_dummy_candidates) == 0:
    # Fallback to raw text (rare; e.g., if drop_first removed the only 3PT category name)
    print("Warning: Could not find a 'shot_type_...3PT...' dummy. Falling back to raw text parsing.")
    df_enc["shot_points"] = df["shot_type"].str.contains("3PT", na=False).astype(int)
    df_enc["shot_points"] = df_enc["shot_points"].replace({1: 3, 0: 2})
else:
    three_col = three_dummy_candidates[0]
    df_enc["shot_points"] = df_enc[three_col] * 3 + (1 - df_enc[three_col]) * 2

# Predicted make probability for all rows
df_enc["make_prob"] = model.predict_proba(X)[:, 1]

# Expected & actual points
df_enc["expected_points"] = df_enc["make_prob"] * df_enc["shot_points"]
df_enc["actual_points"] = df_enc[target_col] * df_enc["shot_points"]

df_enc[["player_name", "shot_points", "make_prob", "expected_points", "actual_points"]].head()


In [ ]:

player_summary = df_enc.groupby("player_name").agg(
    exp_points=("expected_points", "sum"),
    actual_points=("actual_points", "sum"),
    shots=(target_col, "count")
).sort_index()

player_summary["luck"] = player_summary["actual_points"] - player_summary["exp_points"]

print("Top 10 luckiest (actual - expected):")
display(player_summary.sort_values("luck", ascending=False).head(10))

print("\nTop 10 unluckiest:")
display(player_summary.sort_values("luck", ascending=True).head(10))

# Save CSV for download
out_csv = "/mnt/data/player_luck_summary.csv"
player_summary.to_csv(out_csv)
out_csv



### Optional: Team/Game-Level Aggregation

You can repeat a similar aggregation at the game or team level by grouping on
`['team_name', 'espn_game_id']` (or date) to build features for win prediction.
